# 01 — Exploración de datos (EDA)

Exploración de los 4 datasets de MediWatch. Requiere haber corrido `python -m src.data_pipeline.run_all` (genera `data/raw/` y `data/processed/`).

In [ ]:
# Ejecutar desde la raiz del repo (o ajustar la ruta)
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd
pd.set_option("display.max_columns", 40)

In [ ]:
from src.common import DATA_RAW, DATA_PROCESSED
import glob

# Ultimos crudos ingestados
for nombre in ["cum", "vitales", "precios_regulados"]:
    archivos = sorted(glob.glob(str(DATA_RAW / f"{nombre}_*.csv")))
    df = pd.read_csv(archivos[-1], low_memory=False)
    print(f"{nombre}: {df.shape[0]:,} filas x {df.shape[1]} columnas  ({Path(archivos[-1]).name})")

## Hallazgo crítico: no hay llave directa CUM ↔ Vitales

In [ ]:
cum = pd.read_csv(sorted(glob.glob(str(DATA_RAW / "cum_*.csv")))[-1], low_memory=False)
print("¿Columna IUM en el CUM de la API?:", "ium" in cum.columns)
# En el CSV manual la columna existe pero esta 100% vacia; la API ni la expone.
# => la integracion debe hacerse por principio activo (ver 02 y src/data_pipeline/integrate.py)

In [ ]:
vit = pd.read_csv(sorted(glob.glob(str(DATA_RAW / "vitales_*.csv")))[-1], low_memory=False)
print("Tipos de solicitud:")
print(vit["tipo_de_solicitud"].value_counts().head(10) if "tipo_de_solicitud" in vit.columns else vit.filter(like="tipo").iloc[:, 0].value_counts().head(10))

In [ ]:
# Calidad: nulos por columna (top 10) en Vitales
vit.isna().mean().sort_values(ascending=False).head(10).round(3)

## Bases limpias exportadas por el pipeline

In [ ]:
for sub in ["cum", "vitales", "precios", "integracion"]:
    for f in sorted((DATA_PROCESSED / sub).glob("*.csv")):
        print(f"{sub}/{f.name}: {sum(1 for _ in open(f, encoding='utf-8')) - 1:,} filas")